# 03 · LIAR2 — conhecendo o dataset e caçando armadilhas

**O que é o LIAR2?** Um conjunto de ~23 mil *afirmações* (frases curtas ditas por políticos, autoridades etc.) que foram checadas por profissionais e receberam uma nota de verdade. É em **inglês**.

**O que vamos fazer aqui, em 3 passos:**
1. Olhar como as notas de verdade se distribuem.
2. Descobrir quais colunas do dataset são "cola" (dão a resposta de graça) — isso se chama **leakage** (vazamento).
3. Descobrir qual é a dificuldade REAL do problema, usando só a informação limpa.

**As colunas que importam (glossário):**
- `statement` = a afirmação em si (ex: "o desemprego caiu 50%"). **É o que queremos julgar.**
- `label` = a nota de verdade dada pelo checador, de 0 a 5.
- `justification` = o TEXTO que o checador escreveu explicando a nota (cuidado: costuma já dizer o veredito).
- `*_counts` (true_counts, false_counts...) = o histórico do falante — quantas vezes ELE já falou verdade/mentira antes.

**As 6 notas (label):** `0=pants-fire` (mentira deslavada), `1=false`, `2=barely-true` (quase falso), `3=half-true`, `4=mostly-true`, `5=true`.

## Passo 1 · Carregar e olhar a distribuição das notas

Vamos abrir os arquivos e ver quantas afirmações há de cada nota. Depois vamos **binarizar**: transformar as 6 notas em apenas 2 grupos, pra simplificar.

> **Binarizar** = reduzir muitas categorias a duas. Aqui: notas 3, 4, 5 viram **verdadeiro (1)**; notas 0, 1, 2 viram **falso (0)**. Facilita começar e comparar.

> Usamos `train` (pra aprender) e `valid` (pra avaliar). O `test` fica **lacrado** e só é aberto no fim — senão a gente se auto-engana.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

D = '../data/raw/liar2/liar2'
tr = pd.read_csv(f'{D}/train.csv')
va = pd.read_csv(f'{D}/valid.csv')
label_map = {0:'pants-fire',1:'false',2:'barely-true',3:'half-true',4:'mostly-true',5:'true'}
print('afirmacoes no train:', len(tr), '| no valid:', len(va))
print('\nQuantas de cada nota (train):')
print(tr['label'].map(label_map).value_counts())

# binarizar: nota >= 3 -> verdadeiro (1); nota <= 2 -> falso (0)
for d in (tr, va):
    d['bin'] = (d['label'] >= 3).astype(int)
print('\nDepois de binarizar (train): 0=falso, 1=verdadeiro')
print(tr['bin'].value_counts(normalize=True).round(3).to_dict())

### Como ler o que apareceu
- A nota `false` é a mais comum; `true` a mais rara.
- No binário, deu algo perto de **58% falso / 42% verdadeiro**. Guarde o número maior (58%): ele é o **baseline de chute** — se um modeloburro sempre respondesse "falso", acertaria 58%. Qualquer modelo de verdade tem que passar disso.

## Passo 2 · Caçar a "cola" (leakage)

**Leakage (vazamento)** = quando uma coluna entrega a resposta de graça. O modelo parece genial, mas só está copiando a cola — e no mundo real, onde a cola não existe, ele quebra.

Para testar, vamos treinar **3 modelinhos**, cada um enxergando SÓ uma informação, e ver quem acerta mais na validação:
- **(A) só os `*_counts`** — a ficha corrida do falante.
- **(B) só a `justification`** — o texto do checador.
- **(C) só o `statement`** — a afirmação pura. **Esse é o jeito honesto.**

Se A ou B acertam MUITO mais que C, então são cola.

**Ferramentas que vamos usar (explicadas):**
- **TF-IDF** = um jeito de transformar texto em números. Ele dá peso alto às palavras que são "típicas" de um texto e baixas às comuns (tipo "o", "de"). Vira uma tabela que o modelo entende.
- **Regressão Logística (LogReg)** = um classificador simples e rápido que aprende quais palavras puxam pra "falso" ou "verdadeiro". Ótimo ponto de partida.
- **`class_weight='balanced'`** = avisa o modelo pra não preguiçar na classe maior; dá mais atenção à classe menor.

**As duas métricas que vamos olhar:**
- **Acurácia (acc)** = dos casos totais, quantos ele acertou. Simples, mas engana quando as classes são desbalanceadas.
- **F1** = uma nota que equilibra dois erros: pegar falso quando não é, e deixar passar o que é. Vai de 0 a 1; quanto maior, melhor. É mais honesta que a acurácia.

In [ ]:
ytr, yva = tr['bin'], va['bin']
counts_cols = ['true_counts','mostly_true_counts','half_true_counts',
               'mostly_false_counts','false_counts','pants_on_fire_counts']

def acuracia_por_texto(col):
    """Treina um LogReg usando SO uma coluna de texto (via TF-IDF) e avalia na validacao."""
    vec = TfidfVectorizer(max_features=20000, ngram_range=(1,2))  # ate pares de palavras
    Xtr = vec.fit_transform(tr[col].fillna('').astype(str))       # aprende o vocabulario no train
    Xva = vec.transform(va[col].fillna('').astype(str))           # aplica o mesmo no valid
    modelo = LogisticRegression(max_iter=1000, class_weight='balanced').fit(Xtr, ytr)
    pred = modelo.predict(Xva)
    return accuracy_score(yva, pred), f1_score(yva, pred)

# (A) so a ficha corrida (numeros, nao texto)
mA = LogisticRegression(max_iter=1000, class_weight='balanced').fit(tr[counts_cols].fillna(0), ytr)
predA = mA.predict(va[counts_cols].fillna(0))
accA, f1A = accuracy_score(yva, predA), f1_score(yva, predA)
# (B) so a justificativa do checador  e  (C) so a afirmacao
accB, f1B = acuracia_por_texto('justification')
accC, f1C = acuracia_por_texto('statement')

print(f'(A) so a ficha corrida (*_counts) -> acerto {accA:.1%} | F1 {f1A:.3f}')
print(f'(B) so a justificativa            -> acerto {accB:.1%} | F1 {f1B:.3f}')
print(f'(C) so a afirmacao (statement)    -> acerto {accC:.1%} | F1 {f1C:.3f}   <- HONESTO')

## Passo 3 · Lendo o resultado (a conclusão)

Compare os 3 números com o **baseline de chute (~58%)**:

| Modelo | Enxerga | Acerto (esperado) | Veredito |
|---|---|---|---|
| Chute | nada (só a classe maior) | ~58% | piso |
| (A) ficha corrida | histórico do falante | ~63% | fraco, mal passa do chute |
| (C) afirmação | o texto da alegação | ~70% | **este é o número honesto** |
| (B) justificativa | o texto do checador | ~81% | 🚩 **cola!** entrega o veredito |

**Conclusões:**
1. **`justification` é cola** — acerta muito mais que a afirmação pura, porque o checador já escreveu a resposta ali. **Nunca usar como feature.**
2. **`*_counts` é fraco** — mal supera o chute e fica abaixo da afirmação. Não ajuda, e conceitualmente julga o FALANTE, não a alegação. Fora também.
3. **`statement` (~70%) é a dificuldade real.** É daqui que a gente parte pra melhorar de forma honesta.

> **Regra de treino definida:** usar o `statement` (no máximo somando `speaker`/`subject`/`context` como pistas extras). Proibido `justification` e `*_counts`.

**Próximo passo (notebook 04):** subir a "escada de modelos" partindo do statement — começar simples (TF-IDF + Regressão Logística) e depois um modelo mais forte (transformer em inglês, tipo DistilBERT), sempre medindo na validação pra ver se vale a pena.